In [33]:
import collections
import math
import random
import sys
import time
import os
import numpy as np
import torch
from torch import nn
import torch.utils.data as Data
import json
from tqdm import tqdm

In [34]:
import collections
import math
import random
import sys
import time
import os
import numpy as np
import torch
from torch import nn
import torch.utils.data as Data
import json
from tqdm import tqdm

# Train data processing
files_train = os.listdir('/kaggle/input/ea-rgcn-timestamp-pre-w2v/SG_Pre_W2V_Train/SG_Pre_W2V_Train/SG/graph_series/')
path_train = '/kaggle/input/ea-rgcn-timestamp-pre-w2v/SG_Pre_W2V_Train/SG_Pre_W2V_Train/SG/graph_series/'

def getTokens_train():
    tokens = {}
    with open('/kaggle/input/ea-rgcn-timestamp-pre-w2v/SG_Pre_W2V_Train/SG_Pre_W2V_Train/nodes-SG.json') as f:
        tokens = json.load(f)
    return tokens

def getInputSeries_train():
    tokens = getTokens_train()
    nodes_name = list(tokens.keys())
    nodes_int = [tokens[item] for item in tokens.keys()]
    one_hot = np.eye(len(tokens.keys()))
    inputSeries = []
    allSeries = []
    i = 1
    for file in tqdm(files_train):
        vec = np.loadtxt(path_train+file,dtype=np.int32)
        inputSeries.append(vec.tolist())
    return inputSeries

inputSeries_train = getInputSeries_train()

# Test data processing
files_test = os.listdir('/kaggle/input/ea-rgcn-timestamp-pre-w2v/SG_Pre_W2V_Test/SG_Pre_W2V_Test/SG/graph_series/')
path_test = '/kaggle/input/ea-rgcn-timestamp-pre-w2v/SG_Pre_W2V_Test/SG_Pre_W2V_Test/SG/graph_series/'

def getTokens_test():
    tokens = {}
    with open('/kaggle/input/ea-rgcn-timestamp-pre-w2v/SG_Pre_W2V_Test/SG_Pre_W2V_Test/nodes-SG.json') as f:
        tokens = json.load(f)
    return tokens

def getInputSeries_test():
    tokens = getTokens_test()
    nodes_name = list(tokens.keys())
    nodes_int = [tokens[item] for item in tokens.keys()]
    one_hot = np.eye(len(tokens.keys()))
    inputSeries = []
    allSeries = []
    i = 1
    for file in tqdm(files_test):
        vec = np.loadtxt(path_test+file,dtype=np.int32)
        inputSeries.append(vec.tolist())
    return inputSeries

inputSeries_test = getInputSeries_test()

# Fix integer items in train data
inputSeries_train_corrected = [] 
for st in inputSeries_train:
    if isinstance(st, int):
        if st == 0:
            continue
        print(f"Converting integer to list: {st}")
        inputSeries_train_corrected.append([st])
    elif isinstance(st, list):
        inputSeries_train_corrected.append(st)
    else:
        print(f"Warning: Element is neither int nor list. Skipping: {st}")
        pass

# Fix integer items in test data
inputSeries_test_corrected = []
for st in inputSeries_test:
    if isinstance(st, int):
        if st == 0:
            continue
        print(f"Converting integer to list: {st}")
        inputSeries_test_corrected.append([st])
    elif isinstance(st, list):
        inputSeries_test_corrected.append(st)
    else:
        print(f"Warning: Element is neither int nor list. Skipping: {st}")
        pass

# Process train data
counter_train = collections.Counter([tk for st in inputSeries_train_corrected for tk in st])
counter_train = dict(filter(lambda x: x[1] >= 0, counter_train.items()))

idx_to_token_train = [tk for tk, _ in counter_train.items()]
token_to_idx_train = {tk: idx for idx, tk in enumerate(idx_to_token_train)}
dataset_train = [[token_to_idx_train[tk] for tk in st if tk in token_to_idx_train] for st in inputSeries_train_corrected]
num_tokens_train = sum([len(st) for st in dataset_train])

print(f"Train tokens: {num_tokens_train}")
print(f"Train vocabulary size: {len(counter_train)}")

# Process test data
counter_test = collections.Counter([tk for st in inputSeries_test_corrected for tk in st])
counter_test = dict(filter(lambda x: x[1] >= 0, counter_test.items()))

idx_to_token_test = [tk for tk, _ in counter_test.items()]
token_to_idx_test = {tk: idx for idx, tk in enumerate(idx_to_token_test)}
dataset_test = [[token_to_idx_test[tk] for tk in st if tk in token_to_idx_test] for st in inputSeries_test_corrected]
num_tokens_test = sum([len(st) for st in dataset_test])

print(f"Test tokens: {num_tokens_test}")
print(f"Test vocabulary size: {len(counter_test)}")

# Subsampling for train data
def discard_train(idx):
    return random.uniform(0, 1) < 1 - math.sqrt(1e-4 / counter_train[idx_to_token_train[idx]] * num_tokens_train)

subsampled_dataset_train = [[tk for tk in st if not discard_train(tk)] for st in dataset_train]
print(f"Train tokens after subsampling: {sum([len(st) for st in subsampled_dataset_train])}")

# Subsampling for test data
def discard_test(idx):
    return random.uniform(0, 1) < 1 - math.sqrt(1e-4 / counter_test[idx_to_token_test[idx]] * num_tokens_test)

subsampled_dataset_test = [[tk for tk in st if not discard_test(tk)] for st in dataset_test]
print(f"Test tokens after subsampling: {sum([len(st) for st in subsampled_dataset_test])}")

# Generate centers and contexts for train data
def get_centers_and_contexts(dataset, max_window_size):
    centers, contexts = [], []
    for st in dataset:
        if len(st) < 2: # Need at least two words
            continue
        centers += st
        for center_i in range(len(st)):
            window_size = random.randint(1, max_window_size)
            indices = list(range(max(0, center_i - window_size),
            min(len(st), center_i + 1 + window_size)))
            indices.remove(center_i) # Remove center word
            contexts.append([st[idx] for idx in indices])
    return centers, contexts

# Train data centers and contexts
all_centers_train, all_contexts_train = get_centers_and_contexts(subsampled_dataset_train, 5)

# Test data centers and contexts
all_centers_test, all_contexts_test = get_centers_and_contexts(subsampled_dataset_test, 5)

# Generate negatives for train data
def get_negatives(all_contexts, sampling_weights, K):
    all_negatives, neg_candidates, i = [], [], 0
    population = list(range(len(sampling_weights)))
    for contexts in all_contexts:
        negatives = []
        while len(negatives) < len(contexts) * K:
            if i == len(neg_candidates):
                 i, neg_candidates = 0, random.choices(population, sampling_weights, k=int(1e5))
            neg, i = neg_candidates[i], i + 1
            if neg not in set(contexts):
                negatives.append(neg)
        all_negatives.append(negatives)
    return all_negatives

# Generate train negatives
sampling_weights_train = [counter_train[w]**0.75 for w in idx_to_token_train]
all_negatives_train = get_negatives(all_contexts_train, sampling_weights_train, 5)

# Generate test negatives
sampling_weights_test = [counter_test[w]**0.75 for w in idx_to_token_test]
all_negatives_test = get_negatives(all_contexts_test, sampling_weights_test, 5)

# Dataset class
class MyDataset(torch.utils.data.Dataset):
    def __init__(self, centers, contexts, negatives):
        assert len(centers) == len(contexts) == len(negatives)
        self.centers = centers
        self.contexts = contexts
        self.negatives = negatives

    def __getitem__(self, index):
         return (self.centers[index], self.contexts[index], self.negatives[index])

    def __len__(self):
         return len(self.centers)

# Batchify function (Keep original logic)
def batchify(data):
    max_len = max(len(c) + len(n) for _, c, n in data)
    centers, contexts_negatives, masks, labels = [], [], [], []
    for center, context, negative in data:
        cur_len = len(context) + len(negative)
        centers += [center]
        contexts_negatives += [context + negative + [0] * (max_len - cur_len)]
        masks += [[1] * cur_len + [0] * (max_len - cur_len)]
        labels += [[1] * len(context) + [0] * (max_len - len(context))]
    return (torch.tensor(centers).view(-1, 1),
           torch.tensor(contexts_negatives),
           torch.tensor(masks), torch.tensor(labels))

# Create train dataset
dataset_train = MyDataset(all_centers_train, all_contexts_train, all_negatives_train)
batch_size = 512
num_workers = 0 if sys.platform.startswith('win32') else 4
data_iter_train = Data.DataLoader(dataset_train, batch_size, shuffle=True,
                          collate_fn=batchify, num_workers=num_workers)

# Create test dataset
dataset_test = MyDataset(all_centers_test, all_contexts_test, all_negatives_test)
data_iter_test = Data.DataLoader(dataset_test, batch_size, shuffle=True,
                          collate_fn=batchify, num_workers=num_workers)

# Print batch information for train data
for batch in data_iter_train:
    for name, data in zip(['centers', 'contexts_negatives', 'masks', 'labels'], batch):
        print(f"Train {name} shape:", data.shape)
    break

# Print batch information for test data
for batch in data_iter_test:
    for name, data in zip(['centers', 'contexts_negatives', 'masks', 'labels'], batch):
        print(f"Test {name} shape:", data.shape)
    break

# Skip-gram model
def skip_gram(center, contexts_and_negatives, embed_v, embed_u):
    v = embed_v(center)
    u = embed_u(contexts_and_negatives)
    pred = torch.bmm(v, u.permute(0, 2, 1))
    return pred

# Loss function
class SigmoidBinaryCrossEntropyLoss(nn.Module):
    def __init__(self): # none mean sum
        super(SigmoidBinaryCrossEntropyLoss, self).__init__()

    def forward(self, inputs, targets, mask=None):
        inputs, targets, mask = inputs.float(), targets.float(), mask.float()
        res = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction="none", weight=mask)
        return res.mean(dim=1)

loss = SigmoidBinaryCrossEntropyLoss()

# Train model
def train_model(net, lr, num_epochs, data_iter, vocab_size):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print("Training on", device)
    net = net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    for epoch in range(num_epochs):
        start, l_sum, n = time.time(), 0.0, 0
        for batch in data_iter:
            center, context_negative, mask, label = [d.to(device) for d in batch]
            pred = skip_gram(center, context_negative, net[0], net[1])

            l = (loss(pred.view(label.shape), label, mask) *
            mask.shape[1] / mask.float().sum(dim=1)).mean()
            optimizer.zero_grad()
            l.backward()
            optimizer.step()
            l_sum += l.item()
            n += 1
        print('epoch %d, loss %.2f, time %.2fs' % (epoch + 1, l_sum / n, time.time() - start))
    return net

# Create and train train embeddings
embed_size = 100
net_train = nn.Sequential(
    nn.Embedding(num_embeddings=len(idx_to_token_train), embedding_dim=embed_size),
    nn.Embedding(num_embeddings=len(idx_to_token_train), embedding_dim=embed_size)
)

print("Training model on train data...")
trained_net_train = train_model(net_train, 0.01, 15, data_iter_train, len(idx_to_token_train))

# Create and train test embeddings
net_test = nn.Sequential(
    nn.Embedding(num_embeddings=len(idx_to_token_test), embedding_dim=embed_size),
    nn.Embedding(num_embeddings=len(idx_to_token_test), embedding_dim=embed_size)
)

print("Training model on test data...")
trained_net_test = train_model(net_test, 0.01, 15, data_iter_test, len(idx_to_token_test))

# Save train embeddings
train_weight = trained_net_train[0].weight.data.cpu().numpy()
np.savetxt('/kaggle/working/train_word_vectors.txt', train_weight)
np.savetxt('/kaggle/working/train_word_index.txt', np.array(idx_to_token_train, dtype=np.int32), fmt="%d")

# Save test embeddings
test_weight = trained_net_test[0].weight.data.cpu().numpy()
np.savetxt('/kaggle/working/test_word_vectors.txt', test_weight)
np.savetxt('/kaggle/working/test_word_index.txt', np.array(idx_to_token_test, dtype=np.int32), fmt="%d")

print(f"Train embeddings shape: {train_weight.shape}")
print(f"Test embeddings shape: {test_weight.shape}")

100%|██████████| 589/589 [00:01<00:00, 314.56it/s]


Train tokens: 8026671
Train vocabulary size: 865
Test tokens: 6376525
Test vocabulary size: 809
Train tokens after subsampling: 1141290
Test tokens after subsampling: 902259
Train centers shape: torch.Size([512, 1])
Train contexts_negatives shape: torch.Size([512, 60])
Train masks shape: torch.Size([512, 60])
Train labels shape: torch.Size([512, 60])
Test centers shape: torch.Size([512, 1])
Test contexts_negatives shape: torch.Size([512, 60])
Test masks shape: torch.Size([512, 60])
Test labels shape: torch.Size([512, 60])
Training model on train data...
Training on cuda
epoch 1, loss 0.40, time 30.04s
epoch 2, loss 0.26, time 27.39s
epoch 3, loss 0.26, time 25.69s
epoch 4, loss 0.26, time 25.65s
epoch 5, loss 0.25, time 25.69s
epoch 6, loss 0.25, time 25.61s
epoch 7, loss 0.25, time 25.07s
epoch 8, loss 0.25, time 25.75s
epoch 9, loss 0.25, time 25.61s
epoch 10, loss 0.25, time 25.71s
epoch 11, loss 0.25, time 25.41s
epoch 12, loss 0.25, time 25.21s
epoch 13, loss 0.25, time 25.50s
epo